#**Cross-Lingual Transfer Learning for Kinyarwanda Semantic Relatedness
**Objective:**  
This file supplements the main analysis notebook (name 'Kinyarwanda_SemRel.ipynb'). The code for the translation from english to Kinyarwanda (section 3.2. in the main notebook) is provided in this notebook.


In [ ]:
# 1. LOAD (or INSTALL) LIBRARIES

!pip install -q sentence-transformers transformers peft accelerate bitsandbytes pandas numpy scipy matplotlib seaborn

from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np

from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.models import Transformer
import os

In [ ]:
#2. LOAD SemRel2024 DATASET

# Load data from huggingface
data1 = load_dataset("SemRel/SemRel2024", 'eng')
data2 = load_dataset("SemRel/SemRel2024", 'kin')

# Converting to pandas dataframes
semrel_en_train = data1["train"].to_pandas().rename(columns={"label": "human_score"})
semrel_en_test = data1["test"].to_pandas().rename(columns={"label": "human_score"})
semrel_kin_train = data2["train"].to_pandas().rename(columns={"label": "human_score"})
semrel_kin_test = data2["test"].to_pandas().rename(columns={"label": "human_score"})

# Making sure to keep only the three relevant columns
cols = ["sentence1", "sentence2", "human_score"]
semrel_en_train = semrel_en_train[cols]
semrel_en_test = semrel_en_test[cols]
semrel_kin_train = semrel_kin_train[cols]
semrel_kin_test = semrel_kin_test[cols]

# Only using a sample of the english dataset to ensure size consistency
semrel_en_test_sample = semrel_en_test.sample(n=len(semrel_kin_test), random_state=42)
semrel_en_train_sample = semrel_en_train.sample(n=len(semrel_kin_train), random_state=42)

# Checking size consistency
print(f"English sample sizes — train: {len(semrel_en_train_sample)}, test: {len(semrel_en_test_sample)}")
print(f"Kinyarwanda sizes — train: {len(semrel_kin_train)}, test: {len(semrel_kin_test)}")

In [ ]:
#3. LOAD TRANSLATION MODEL
  #The chosen transaltion model uses the FLORES BENCHMARK

model_name = "facebook/nllb-200-distilled-600M"

# Load model and tokenizer (explicit)
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn", use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

# Helper: find forced_bos_token_id for a language code robustly
def get_lang_token_id(tokenizer, model, tgt_lang):
    # 1) Preferred: tokenizer.lang_code_to_id (exists for many NLLB releases)
    if hasattr(tokenizer, "lang_code_to_id") and isinstance(tokenizer.lang_code_to_id, dict):
        if tgt_lang in tokenizer.lang_code_to_id:
            return tokenizer.lang_code_to_id[tgt_lang]

    # 2) Some models store mapping in model.config.lang_code_to_id
    cfg_map = getattr(model.config, "lang_code_to_id", None)
    if isinstance(cfg_map, dict) and tgt_lang in cfg_map:
        return cfg_map[tgt_lang]

    # 3) Try to find a matching special token like "<kin_Latn>" in additional_special_tokens
    #    and convert it to id.
    for tok in getattr(tokenizer, "additional_special_tokens", []):
        if tgt_lang in tok:
            tok_id = tokenizer.convert_tokens_to_ids(tok)
            if tok_id is not None:
                return tok_id

    # 4) As a last resort, search all tokens for a substring match (slower)
    #    (This is rarely necessary but safe.)
    vocab = tokenizer.get_vocab()
    for tok, idx in vocab.items():
        if tgt_lang in tok:
            return idx

    # If nothing found, return None so caller can handle it
    return None

# Example: check we can find kin token
tgt = "kin_Latn"
lang_id = get_lang_token_id(tokenizer, model, tgt)
print("Resolved token id for", tgt, "->", lang_id)
if lang_id is None:
    raise RuntimeError("Could not resolve language token id for " + tgt +
                       ". Inspect tokenizer.special_tokens_map and tokenizer.additional_special_tokens.")

# Robust translation function using the resolved id
def translate_batch(texts, src_lang="eng_Latn", tgt_lang="kin_Latn", max_length=256):
    # ensure small batches (avoid OOM)
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # set src_lang on tokenizer when required (some tokenizers use this attribute)
    try:
        tokenizer.src_lang = src_lang
    except Exception:
        pass

    forced_id = get_lang_token_id(tokenizer, model, tgt_lang)
    if forced_id is None:
        raise RuntimeError(f"Language id for {tgt_lang} not found; cannot force target language.")

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            forced_bos_token_id=forced_id,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return decoded

In [ ]:
#4. TRANSLATE SEMREL ENGLISH PAIRS TO CREATE SYNTHETIC KINYARWANDA DATASET

translated_s1, translated_s2 = [], []

batch_size = 64

for i in tqdm(range(0, len(semrel_en_train), batch_size)):
    batch_s1 = semrel_en_train_sample["sentence1"].iloc[i:i+batch_size].tolist()
    batch_s2 = semrel_en_train_sample["sentence2"].iloc[i:i+batch_size].tolist()

    kin_s1 = translate_batch(batch_s1)
    kin_s2 = translate_batch(batch_s2)

    translated_s1.extend(kin_s1)
    translated_s2.extend(kin_s2)

# Build synthetic (translated) dataset
semrel_kin_aug = pd.DataFrame({
    "sentence1": translated_s1,
    "sentence2": translated_s2,
    "human_score": semrel_en_train["human_score"]
})

In [ ]:
semrel_kin_aug.head()